In [ ]:
%matplotlib inline

In [ ]:

import warnings
import os
import pandas as pd
import seaborn as sns
import numpy as np
import networkx as netx
import pywt
import matplotlib.pyplot as plt
import matplotlib.style as style
import matplotlib.cm as cm
from scipy.io import loadmat
from scipy.spatial.distance import cdist
import scipy
from sklearn.manifold import LocallyLinearEmbedding
import nilearn as nl             # pip: nilearn==0.10.0
from nilearn.maskers import NiftiMasker
import nibabel as nib
import nltools as nlt
from matplotlib.colors import ListedColormap
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from itertools import compress
from scipy.optimize import minimize


In [ ]:

def nii2cmu(nifti_file, mask_file=None):
    '''
    inputs:
      nifti_file: a filename of a .nii or .nii.gz file to be converted into
                  CMU format

      mask_file: a filename of a .nii or .nii.gz file to be used as a mask; all
                 zero-valued voxels in the mask will be ignored in the CMU-
                 formatted output.  If ignored or set to None, no voxels will
                 be masked out.

    outputs:
      Y: a number-of-timepoints by number-of-voxels numpy array containing the
         image data.  Each row of Y is an fMRI volume in the original nifti
         file.

      R: a number-of-voxels by 3 numpy array containing the voxel locations.
         Row indices of R match the column indices in Y.
    '''
    def fullfact(dims):
        '''
        Replicates MATLAB's fullfact function (behaves the same way)
        '''
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        else:
            aftervals = np.asmatrix(fullfact(dims[1:]))
            inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
            row = 0
            for i in range(aftervals.shape[0]):
                inds[row:(row + len(vals)), 0] = vals
                inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
                row += len(vals)
            return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        if type(nifti_file) == str:
            img = nib.load(nifti_file)
        elif type(nifti_file) == nib.nifti1.Nifti1Image:
            img = nifti_file
        else:
            raise ValueError('nifti_file must be a filename or nibabel Nifti1Image object')

        mask = NiftiMasker(mask_strategy='background')
        if mask_file is None:
            mask.fit(nifti_file)
        else:
            mask.fit(mask_file)

    hdr = img.header
    S = img.get_sform()
    vox_size = hdr.get_zooms()
    im_size = img.shape

    if len(img.shape) > 3:
        N = img.shape[3]
    else:
        N = 1

    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1]-1

    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]

    return {'Y': Y, 'R': R}

def cmu2nii(Y, R, template=None):
    '''
    inputs:
      Y: a number-of-timepoints by number-of-voxels numpy array containing the
         image data.  Each row of Y is an fMRI volume in the original nifti
         file.

      R: a number-of-voxels by 3 numpy array containing the voxel locations.
         Row indices of R match the column indices in Y.

      template: a filename of a .nii or .nii.gz file to be used as an image
                template.  Header information of the outputted nifti images will
                be read from the header file.  If this argument is ignored or
                set to None, header information will be inferred based on the
                R array.

    outputs:
      nifti_file: a filename of a .nii or .nii.gz file to be converted into
                  CMU format

      mask_file: a filename for a .nii or .nii.gz file to be used as a mask; all
                 zero-valued voxels in the mask will be ignored in the CMU-
                 formatted output

    outputs:
      img: a nibabel Nifti1Image object containing the fMRI data
    '''
    Y = np.array(Y, ndmin=2)
    img = nib.load(template)
    S = img.affine
    locs = np.array(np.dot(R - S[:3, 3], np.linalg.inv(S[0:3, 0:3])), dtype='int')

    data = np.zeros(tuple(list(img.shape)[0:3]+[Y.shape[0]]))

    # loop over data and locations to fill in activations
    for i in range(Y.shape[0]):
        for j in range(R.shape[0]):
            data[locs[j, 0], locs[j, 1], locs[j, 2], i] = Y[i, j]

    return nib.Nifti1Image(data, affine=img.affine)
    

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)

        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes) + 1)]
        labels.append(np.argmax(label_weights) + 1)

    return pd.DataFrame({'code': labels, 'Network': [list(lookup_table.values())[i - 1] for i in labels]})

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def purity(y_clust,y_class):

    size_clust = np.max(y_clust)
    len_clust = len(y_clust)
    clusters_labels_try = [None] * size_clust

    for i in range(len_clust):
        index = y_clust[i]
    
        if clusters_labels_try[index-1] is None:
            clusters_labels_try[index-1] = y_class[i]

        else:
            clusters_labels_try[index-1] = np.hstack((clusters_labels_try[index-1], y_class[i]))
    purity = 0
    for c in clusters_labels_try:
        c = c.astype(int)
        if isinstance(c, np.int64):
    
            c = np.array([c]) 
        y = np.bincount(c) #I find occurrences of the present elements
        maximum = np.max(y) #I take the item more frequently
        purity += maximum

    purity = purity/len_clust

    return purity

# Pie Man Data

## Load the data here

The data we're using for this, to start, is fMRI data from particpants scanned while being told a story in different listening conditions.  The story was told either intact, or scrambled by the paragraph, or scrambled by the word, or just while the participant was quietly resting.  

The fMRI data was then reduced using Hierarchicaly topographic factor analysis (HTFA which is a Bayesian factor analysis model designed to analyze brain network dynamics in multi-subject datasets. It builds upon Topographic Factor Analysis (TFA), extending it to incorporate data from multiple subjects, allowing for the identification of shared brain network hubs and the comparison of individual subject's brain activity against a global template. In this case, the >100,000 voxels was reduced to 700 nodes with varying widths.




In [ ]:
pieman_name = '../data/raw/pieman_data.mat'
pieman_data = loadmat(pieman_name)
pieman_conds = ['intact', 'paragraph', 'word', 'rest']

In [ ]:
posterior = loadmat('../data/raw/pieman_posterior_K700.mat')
centers = posterior['posterior']['centers'][0][0][0][0][0]
widths = np.array(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T))

These lines of code separate the participant data by listening conditions (and skips over one participant with an incorrect number of timepoints in the paragraph condition)

In [ ]:
data = []
conds = []
for c in pieman_conds:
    if c == 'paragraph':
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.where(np.arange(pieman_data[c].shape[1]) != 3)[0]))
    else:
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.arange(pieman_data[c].shape[1])))
    data.extend(next_data)
    conds.extend([c]*len(next_data))


In [ ]:
conds_array = np.array(conds)

First, lets just play with some of the intact listening condition data:

In [ ]:
intact_list = [data[i] for i in np.where(conds_array=='intact')[0]]
intact_array = np.array(intact_list)

In [ ]:
paragraph_list = [data[i] for i in np.where(conds_array=='paragraph')[0]]
paragraph_array = np.array(paragraph_list)

In [ ]:
word_list = [data[i] for i in np.where(conds_array=='word')[0]]
word_array = np.array(word_list)

In [ ]:
rest_list = [data[i] for i in np.where(conds_array=='rest')[0]]
rest_array = np.array(rest_list)

## Network Analysis

Let's load in what we need for our network analysis:

In [ ]:
key = os.path.join('../data/raw/Schaefer2018_1000Parcels_7Networks_order.txt')
key = pd.read_csv(key, sep='\t', header=None, names=['id', 'name', 'x', 'y', 'z', 't']).drop('t', axis=1)
key['study'] = key['name'].apply(lambda x: x.split('_')[0])
key['hemisphere'] = key['name'].apply(lambda x: x.split('_')[1][0])
key['network'] = key['name'].apply(lambda x: x.split('_')[2])
key.drop('name', axis=1, inplace=True)

lookup_table = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor',
    'DorsAttn': 'Dorsal attention',
    'SalVentAttn': 'Ventral attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal',
    'Default': 'Default mode'
}

network_colors = {
    'Visual': '#D7DF23',
    'Somatomotor': '#39B54A',
    'Dorsal attention': '#00A79D',
    'Ventral attention': '#27AAE1',
    'Limbic': '#1C75BC',
    'Frontoparietal': '#92278F',
    'Default mode': '#EE2A7B'
}

colors = ['#888888']
colors.extend([v for k, v in network_colors.items()])
network_cmap = ListedColormap(colors, N=len(colors)*2, name='networks')  # why do I need to double the number of colors?

network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}

key['network'] = key['network'].apply(lambda x: lookup_table[x])
key['code'] = key['network'].apply(lambda x: network_codes[x])
key.set_index('id', inplace=True)
key.loc[0, 'code'] = 0  # append a background (non-network) code
key           # pip: nibabel==5.0.1

In [ ]:
nii_fname = os.path.join('../data/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz')


So this is a plot of the 7 networks in the brain determined by Yeo et al. (2017).  

In [ ]:
nx = nlt.Brain_Data(nii_fname)
nx.data = np.array([key.loc[i, 'code'] for i in nx.data]).astype(float)
networks = nx.to_nifti()
nl.plotting.plot_glass_brain(networks, cmap=network_cmap, display_mode='lyrz')
plt.show()
plt.clf()


We need a way to map our 700 nodes to these networks:

In [ ]:
# load network parcellations in CMU format
networks_cmu = nii2cmu(nii_fname)

# covert voxel values in the reference image to network codes
networks_cmu['Y'] = np.atleast_2d(np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float))

And these are the labels/code for each of those nodes:

In [ ]:
node_codes = node_labels(centers, widths, networks_cmu)
node_codes

Ok and plot that now (we could vary the node size relative to the width, but just for simplicity the nodes are all the same size here).

In [ ]:
nl.plotting.plot_connectome(np.eye(centers.shape[0]), centers, node_size=10, node_color=[colors[i] for i in node_codes['code']], display_mode='lyrz')
plt.show()
plt.clf()

Ok now that we have the network labels for each of those nodes, lets see the overall count for each of those nodes in this dataset:

In [ ]:
fig = plt.figure(figsize=(4, 3))
ax = plt.gca()

sns.histplot(data=node_codes, x='code', color='k', shrink=0.75, hue='code', hue_order=range(1, 8), palette=colors[1:], alpha=1.0, legend=False, ax=ax, discrete=True, edgecolor=None)
ax.set_xlabel('Network', fontsize=12)
ax.set_ylabel('Number of nodes', fontsize=12)

ax.set_xticks([])
ax.spines[['right', 'top']].set_visible(False)
plt.show()
plt.clf()
fig.savefig(os.path.join('htfa_node_counts.pdf'), bbox_inches='tight')

## Across all conditions!

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.ndimage import gaussian_filter1d



################################################################################
# Archetypal Analysis        ####################################################
################################################################################
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

def furthest_sum(X, n_archetypes, random_state=None):
    """
    Memory-safe FurthestSum initialization.
    
    Parameters
    ----------
    X : np.ndarray, shape (n_features, n_samples)
        Data matrix.
    n_archetypes : int
        Number of archetypes to select.
    random_state : int or None
        Seed for reproducibility.
    
    Returns
    -------
    selected : list of int
        Selected indices for initialization.
    """
    rng = np.random.default_rng(random_state)
    n_samples = X.shape[1]
    selected = [rng.integers(0, n_samples)]

    for _ in range(1, n_archetypes):
        dists = np.zeros(n_samples, dtype=np.float32)
        for i in range(n_samples):
            dists[i] = sum(np.linalg.norm(X[:, i] - X[:, j]) for j in selected)
        dists[selected] = -np.inf  # don't re-select
        selected.append(np.argmax(dists))

    return selected
    
class MultiSubjectArchetypalAnalysis(BaseEstimator, TransformerMixin):
    def __init__(self, n_archetypes=3, tmax=20, iterations=10, init_method='furthest', random_state=42, verbose=True):
        self.n_archetypes = n_archetypes
        self.tmax = tmax
        self.iterations = iterations
        self.init_method = init_method
        self.random_state = random_state
        self.verbose = verbose
        self.Z_ = None
        self.A_ = None

    def fit(self, Xs):
        """
        Fit MS-AA to a list of subject matrices.

        Parameters
        ----------
        Xs : list of np.ndarray
            Each with shape (n_features, n_timepoints)
        """
        # Convert to float32 to reduce memory usage
        Xs = [X.astype(np.float32) for X in Xs]
        n_subjects = len(Xs)
        n_features, _ = Xs[0].shape
        k = self.n_archetypes

        # Concatenate all subjects once
        X_concat = np.concatenate(Xs, axis=1).astype(np.float32)

        # Initialize archetypes
        if self.init_method == 'furthest':
            init_indices = furthest_sum(X_concat, k, random_state=self.random_state)
            B = np.zeros((X_concat.shape[1], k), dtype=np.float32)
            for i, idx in enumerate(init_indices):
                B[idx, i] = 1.0
        elif self.init_method == 'random':
            rng = np.random.default_rng(self.random_state)
            init_indices = rng.choice(X_concat.shape[1], size=k, replace=False)
            B = np.zeros((X_concat.shape[1], k), dtype=np.float32)
            for i, idx in enumerate(init_indices):
                B[idx, i] = 1.0
        else:  # identity
            B = np.eye(X_concat.shape[1], k, dtype=np.float32)

        Z = X_concat @ B
        A_subjects = [np.zeros((k, X.shape[1]), dtype=np.float32) for X in Xs]

        for iteration in range(self.iterations):
            # Update per-subject A
            for i, X in enumerate(Xs):
                A_subjects[i] = self._computeA(X, Z, self.tmax)

            # Concatenate for B update
            A_concat = np.concatenate(A_subjects, axis=1).astype(np.float32)
            B = self._computeB(X_concat, A_concat, self.tmax)
            Z = X_concat @ B

            if self.verbose:
                rss_total = sum(self._rss(Xs[i], A_subjects[i], Z) for i in range(n_subjects))
                print(f"Iteration {iteration + 1}, Total RSS: {rss_total:.4f}")

        self.Z_ = Z
        self.A_ = A_subjects
        return self

    def transform(self, X):
        return self._computeA(X.astype(np.float32), self.Z_, self.tmax)

    def inverse_transform(self, A):
        return self.Z_ @ A

    def archetypes(self):
        return self.Z_

    def subject_weights(self):
        return self.A_

    def _computeA(self, X, Z, tmax):
        m, n = X.shape
        k = self.n_archetypes
        A = np.zeros((k, n), dtype=np.float32)
        A[0, :] = 1.0

        for t in range(tmax):
            G = 2.0 * ((Z.T @ Z) @ A - Z.T @ X)
            argmins = np.argmin(G, axis=0)
            e = np.zeros_like(G)
            e[argmins, np.arange(n)] = 1.0
            A += 2.0 / (t + 2.0) * (e - A)
        return A

    def _computeB(self, X, A, tmax):
        k, n = A.shape
        B = np.zeros((n, k), dtype=np.float32)
        B[0, :] = 1.0

        for t in range(tmax):
            t1 = X.T @ (X @ B) @ (A @ A.T)
            t2 = X.T @ (X @ A.T)
            G = 2.0 * (t1 - t2)
            argmins = np.argmin(G, axis=0)
            e = np.zeros_like(G)
            e[argmins, np.arange(k)] = 1.0
            B += 2.0 / (t + 2.0) * (e - B)
        return B

    def _rss(self, X, A, Z):
        return np.linalg.norm(X - Z @ A)
        


def msaa_decoding_pipeline(data, labels, n_archetypes_list, smoothing_sigma=0, verbose=True):
    """
    Multi-subject archetypal analysis decoding pipeline.

    Parameters
    ----------
    data : np.ndarray
        Shape (n_subjects, n_timepoints, n_features)
    labels : np.ndarray
        Shape (n_subjects, n_timepoints) - decoding targets
    n_archetypes_list : list
        List of archetype numbers to try
    smoothing_sigma : float
        Sigma for Gaussian temporal smoothing (0 = no smoothing)
    verbose : bool
        Whether to print progress

    Returns
    -------
    results : dict
        Mapping from n_archetypes -> decoding accuracy
    """

    n_subjects, n_timepoints, n_features = data.shape
    results = {}

    # Reformat to list-of-matrices for MS-AA: (features x time)
    Xs = [data[sub].T for sub in range(n_subjects)]

    for k in n_archetypes_list:
        if verbose:
            print(f"\n=== Testing {k} archetypes ===")

        # Fit MS-AA
        msaa = MultiSubjectArchetypalAnalysis(n_archetypes=k, tmax=20, iterations=10, verbose=False)
        msaa.fit(Xs)

        # Get per-subject weights over time
        A_subjects = msaa.subject_weights()  # list of (k x time)

        # Optionally smooth over time
        if smoothing_sigma > 0:
            A_subjects = [gaussian_filter1d(A, sigma=smoothing_sigma, axis=1) for A in A_subjects]

        # Prepare decoding inputs
        X_all = []
        y_all = []
        groups = []

        for sub_idx, A in enumerate(A_subjects):
            # shape: (time x k)
            A_T = A.T
            X_all.append(A_T)
            y_all.append(labels[sub_idx])
            groups.extend([sub_idx] * n_timepoints)

        X_all = np.vstack(X_all)
        y_all = np.concatenate(y_all)
        groups = np.array(groups)

        # Leave-one-subject-out decoding
        logo = LeaveOneGroupOut()
        preds = np.zeros_like(y_all)

        for train_idx, test_idx in logo.split(X_all, y_all, groups):
            clf = LogisticRegression(max_iter=500)
            clf.fit(X_all[train_idx], y_all[train_idx])
            preds[test_idx] = clf.predict(X_all[test_idx])

        acc = accuracy_score(y_all, preds)
        results[k] = acc

        if verbose:
            print(f"Decoding accuracy: {acc:.4f}")

    return results




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from scipy.ndimage import gaussian_filter1d
from itertools import product

def msaa_decoding_tuning(
    data, labels, groups, n_archetypes_list, smoothing_list, 
    cv_splits=5, verbose=True
):
    """
    Multi-subject Archetypal Analysis decoding pipeline with tuning.
    Now also returns per-fold scores for confidence intervals.
    """
    n_subjects, n_time, n_features = data.shape
    best_score = -np.inf
    best_params = None
    gkf = GroupKFold(n_splits=cv_splits)

    results = []  # store (n_archetypes, sigma, mean, std, scores)

    for n_arch, sigma in product(n_archetypes_list, smoothing_list):
        if verbose:
            print(f"Testing n_archetypes={n_arch}, sigma={sigma}...")

        # Fit Multi-Subject Archetypal Analysis
        Xs = [data[sub].T for sub in range(n_subjects)]  # convert to features x time
        msaa = MultiSubjectArchetypalAnalysis(
            n_archetypes=n_arch, tmax=20, iterations=10, verbose=False
        )
        msaa.fit(Xs)
        A_subjects = msaa.subject_weights()  # list of (k x time)

        # Temporal smoothing
        if sigma > 0:
            A_subjects = [gaussian_filter1d(A, sigma=sigma, axis=1) for A in A_subjects]

        # Prepare decoding data
        X_all = np.vstack([A.T for A in A_subjects])  # (subjects*time, k)
        y_all = np.repeat(labels, n_time)             # (subjects*time,)
        groups_all = np.repeat(groups, n_time)        # (subjects*time,)

        # Classifier
        clf = LogisticRegression(max_iter=500, solver='liblinear')

        # Cross-validation
        scores = cross_val_score(
            clf, X_all, y_all, cv=gkf, groups=groups_all, scoring='accuracy'
        )
        mean_score = np.mean(scores)
        std_score = np.std(scores)

        results.append((n_arch, sigma, mean_score, std_score, scores))

        if mean_score > best_score:
            best_score = mean_score
            best_params = (n_arch, sigma)

        if verbose:
            print(f"  Accuracy = {mean_score:.3f} ± {std_score:.3f}")

    if verbose:
        print("\nBest parameters:")
        print(f"  n_archetypes={best_params[0]}, sigma={best_params[1]}, accuracy={best_score:.3f}")

    return best_params, best_score, results


def plot_msaa_scree(results, ci=95):
    """
    Plot decoding accuracy as a function of n_archetypes, colored by smoothing.
    Adds error bars (mean ± CI).
    """
    z = 1.96 if ci == 95 else 1.0  # z-value for CI, default 95%
    
    results = np.array(results, dtype=object)
    sigmas = sorted(np.unique(results[:,1]))

    plt.figure(figsize=(7,5))
    for sigma in sigmas:
        subset = results[results[:,1] == sigma]
        n_archs = subset[:,0].astype(int)
        means   = subset[:,2].astype(float)
        stds    = subset[:,3].astype(float)

        # Standard error of the mean across folds
        n_folds = len(subset[0,4])
        sems = stds / np.sqrt(n_folds)
        ci_vals = z * sems

        plt.errorbar(
            n_archs, means, yerr=ci_vals,
            marker='o', capsize=4, label=f"sigma={sigma}"
        )

    plt.xlabel("Number of Archetypes (k)", fontsize=12)
    plt.ylabel("Decoding Accuracy", fontsize=12)
    plt.title(f"Scree Plot of Decoding Accuracy (±{ci}% CI)", fontsize=14)
    plt.legend(title="Smoothing σ")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()


In [ ]:
n_subjects, n_time, n_features = 72, 300, 700
labels = np.array([0]*36 + [1]*36)
groups = np.arange(n_subjects)

In [ ]:

data = np.vstack((intact_array, word_array))
best_params_intact_word, best_score_intact_word, results_intact_word = msaa_decoding_tuning(
    data, labels, groups,
    n_archetypes_list=list(range(50, 70, 10)),
    smoothing_list=list(range(4, 6, 2)),
    cv_splits=5
)

plot_msaa_scree(results_intact_word, ci=95)

In [ ]:
data = np.vstack((intact_array, rest_array[:, :300, :]))
best_params_intact_rest, best_score_intact_rest, results_intact_rest = msaa_decoding_tuning(
    data, labels, groups,
    n_archetypes_list=list(range(50, 100, 10)),
    smoothing_list=list(range(4, 8, 2)),
    cv_splits=5
)

plot_msaa_scree(results_intact_rest, ci=95)

In [ ]:
data = np.vstack((word_array, rest_array[:, :300, :]))
best_params_word_rest, best_score_word_rest, results_word_rest = msaa_decoding_tuning(
    data, labels, groups,
    n_archetypes_list=list(range(50, 100, 10)),
    smoothing_list=list(range(4, 8, 2)),
    cv_splits=5
)

plot_msaa_scree(results_word_rest, ci=95)

Ok so that works... but that's a lot of archetypes! I'll need to explore that more... in the meantime, let's see how to map this back on to the brain

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt



################################################################################
# MS-AA Decoding Pipeline
################################################################################
def msaa_decoding_pipeline(data, labels, n_archetypes_list, smoothing_sigma=0, verbose=True):
    n_subjects, n_timepoints, n_features = data.shape
    results = {}

    Xs = [data[sub].T for sub in range(n_subjects)]

    for k in n_archetypes_list:
        if verbose:
            print(f"\n=== Testing {k} archetypes ===")
        msaa = MultiSubjectArchetypalAnalysis(n_archetypes=k, tmax=20, iterations=10, verbose=False)
        msaa.fit(Xs)
        A_subjects = msaa.subject_weights()
        if smoothing_sigma > 0:
            A_subjects = [gaussian_filter1d(A, sigma=smoothing_sigma, axis=1) for A in A_subjects]

        # Prepare data for decoding
        X_all = []
        y_all = []
        groups = []
        for sub_idx, A in enumerate(A_subjects):
            A_T = A.T
            X_all.append(A_T)
            y_all.append(np.full(A.shape[1], labels[sub_idx]))
            groups.extend([sub_idx]*A.shape[1])

        X_all = np.vstack(X_all)
        y_all = np.concatenate(y_all)
        groups = np.array(groups)

        # Leave-one-subject-out decoding
        logo = LeaveOneGroupOut()
        preds = np.zeros_like(y_all)
        for train_idx, test_idx in logo.split(X_all, y_all, groups):
            clf = LogisticRegression(max_iter=500)
            clf.fit(X_all[train_idx], y_all[train_idx])
            preds[test_idx] = clf.predict(X_all[test_idx])
        acc = accuracy_score(y_all, preds)
        results[k] = {'accuracy': acc, 'A_subjects': A_subjects, 'Z': msaa.archetypes()}

        if verbose:
            print(f"Decoding accuracy: {acc:.4f}")
    return results



In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# ---------- Safe best-k picker ----------
def pick_best_k(results):
    if not len(results):
        raise ValueError("`results` is empty. Run msaa_decoding_pipeline first.")
    # results keys are the k's you actually ran (ints)
    k_best = max(results.keys(), key=lambda k: results[k]['accuracy'])
    return k_best

# ---------- Rank archetypes by decoding (uses condition labels) ----------
def rank_archetypes_by_decoding(results, labels_by_subject, n_top=5, k=None):
    """
    labels_by_subject: shape (n_subjects,), e.g., 0/1 condition label per subject.
    If k is None, uses the best k in `results`.
    """
    if k is None:
        k = pick_best_k(results)
    if k not in results:
        raise KeyError(f"k={k} not found in results. Available keys: {list(results.keys())}")

    A_subjects = results[k]['A_subjects']  # list of (k x time)
    n_subjects = len(A_subjects)
    # Build (timepoints*n_subjects) x k design and matching labels
    X_all = np.vstack([A.T for A in A_subjects])  # (sum_time) x k
    y_all = np.concatenate([np.full(A.shape[1], labels_by_subject[s]) for s, A in enumerate(A_subjects)])

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_all, y_all)
    # Importance per archetype (mean abs coef across classes)
    coeffs = np.abs(clf.coef_).mean(axis=0)  # shape (k,)
    top_idxs = np.argsort(coeffs)[::-1][:n_top]
    return {k: top_idxs}, coeffs

# ---------- Extract per-node, time-resolved archetype activity ----------
def extract_archetype_node_activity(results, data, k=None):
    """
    data: (n_subjects, n_timepoints, n_nodes)
    Returns: (n_subjects, k, n_timepoints, n_nodes)
    """
    n_subjects, n_timepoints, n_nodes = data.shape

    if k is None:
        k = pick_best_k(results)
    if k not in results:
        raise KeyError(f"k={k} not found in results. Available keys: {list(results.keys())}")

    Z = results[k]['Z']             # (n_nodes x k)
    A_subjects = results[k]['A_subjects']  # list of (k x time)
    k_inferred = Z.shape[1]
    if k_inferred != k:
        # Use whatever Z says (prevents mismatches if someone passed the wrong k)
        k = k_inferred

    archetype_node_activity = np.zeros((n_subjects, k, n_timepoints, n_nodes), dtype=np.float32)

    for subj_idx in range(n_subjects):
        A = A_subjects[subj_idx]  # (k x time)
        if A.shape[0] != k:
            raise ValueError(f"Subject {subj_idx}: A has {A.shape[0]} archetypes, but Z has {k}.")
        # For each archetype: (nodes x time) = (nodes,)[:,None] * (time,)[None,:]
        # Then transpose to (time x nodes)
        for arch_idx in range(k):
            archetype_node_activity[subj_idx, arch_idx, :, :] = (
                (Z[:, arch_idx][:, None] * A[arch_idx, :][None, :]).T
            )

    return archetype_node_activity




In [ ]:

results = msaa_decoding_pipeline(data, labels, n_archetypes_list=[50,60], smoothing_sigma=6)

# 1) Pick best k that actually exists in results
best_k = pick_best_k(results)
print("Best k (by accuracy):", best_k, "->", results[best_k]['accuracy'])

# 2) Rank archetypes for that k using condition labels per subject
top_archetypes_dict, arch_coeffs = rank_archetypes_by_decoding(results, labels, n_top=5, k=best_k)
print("Top archetypes:", top_archetypes_dict[best_k])

# 3) Extract node×time activity per archetype for that k
archetype_node_activity = extract_archetype_node_activity(results, data, k=best_k)



In [ ]:
best_k = 60
top_archetypes_dict, arch_coeffs = rank_archetypes_by_decoding(results, labels, n_top=5, k=best_k)
print("Top archetypes:", top_archetypes_dict[best_k])
archetype_node_activity_best = extract_archetype_node_activity(results, data, k=60)
best_k = 50
top_archetypes_dict, arch_coeffs = rank_archetypes_by_decoding(results, labels, n_top=5, k=best_k)
print("Top archetypes:", top_archetypes_dict[best_k])
archetype_node_activity_compare = extract_archetype_node_activity(results, data, best_k)

In [ ]:
import imageio

In [ ]:
import numpy as np
from nilearn import plotting
import matplotlib.pyplot as plt
import imageio
import os
import seaborn as sns

def create_condition_comparison_gif(archetype_node_activity, centers, node_codes,
                                    arch_idx=0, n_timepoints=10,
                                    condition_labels=['Intact','Scrambled'],
                                    n_subjects_per_condition=5,
                                    output_dir='comparison_gifs',
                                    cmap='RdBu_r',
                                    colors=None):
    """
    Create a GIF showing archetype dynamics with:
    - both conditions side by side
    - timeline bar
    - histogram of node assignments to Yeo-7 networks

    Parameters
    ----------
    archetype_node_activity : array (subjects x archetypes x time x nodes)
    centers : array (nodes x 3) node coordinates
    node_codes : DataFrame or array mapping nodes -> networks
                 must contain 'code' column with network code (1–7)
    colors : list of hex or RGB colors (length 8, index 0 unused)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Precompute per-condition mean
    cond_means = []
    for cond_idx in range(len(condition_labels)):
        start = cond_idx * n_subjects_per_condition
        end = start + n_subjects_per_condition
        data_cond = archetype_node_activity[start:end, arch_idx, :, :]  # subj x time x nodes
        cond_means.append(np.mean(data_cond, axis=0))  # time x nodes
    
    cond_means = np.array(cond_means)  # cond x time x nodes
    
    # Global colorbar scaling across both conditions and all time
    global_min = np.min(cond_means)
    global_max = np.max(cond_means)
    
    frames = []
    for t in range(n_timepoints):
        fig, axes = plt.subplots(2, len(condition_labels), figsize=(12, 6),
                                 gridspec_kw={'height_ratios': [3, 1]})
        
        # Loop over conditions
        for c, cond_name in enumerate(condition_labels):
            node_weights = cond_means[c, t, :]
            
            # --- Brain markers ---
            plotting.plot_markers(
                node_values=node_weights,
                node_coords=centers,
                node_cmap=cmap,
                node_vmin=global_min,
                node_vmax=global_max,
                title=f"{cond_name} - t={t}",
                display_mode='ortho',
                axes=axes[0, c],
                figure=fig
            )
            
            # --- Histogram of networks (weighted) ---
            node_codes_copy = node_codes.copy()
            node_codes_copy['NetworkWeight'] = node_weights
            
            sns.histplot(
                data=node_codes_copy,
                x='code',
                weights='NetworkWeight',
                shrink=0.75,
                hue='code',
                hue_order=range(1, 8),
                palette=colors[1:] if colors is not None else None,
                alpha=1.0,
                legend=False,
                ax=axes[1, c],
                discrete=True,
                edgecolor=None
            )
            axes[1, c].set_xlabel('Network', fontsize=11)
            axes[1, c].set_ylabel('Weighted activity', fontsize=11)
            axes[1, c].set_ylim(global_min, global_max)
        
        # --- Timeline bar ---
        timeline_ax = fig.add_axes([0.1, 0.02, 0.8, 0.03])
        timeline_ax.plot(range(n_timepoints), np.zeros(n_timepoints), 'k-')
        timeline_ax.scatter(t, 0, c='red', s=50, zorder=5)
        timeline_ax.set_xlim(-0.5, n_timepoints-0.5)
        timeline_ax.set_ylim(-0.5, 0.5)
        timeline_ax.axis('off')
        
        plt.tight_layout(rect=[0, 0.05, 1, 1])
        temp_path = os.path.join(output_dir, f"frame_{t}.png")
        plt.savefig(temp_path, dpi=100)
        plt.close(fig)
        
        frames.append(imageio.imread(temp_path))
    
    gif_path = os.path.join(output_dir, f"comparison_arch{arch_idx}.gif")
    imageio.mimsave(gif_path, frames, duration=0.4)
    
    # Cleanup
    for t in range(n_timepoints):
        os.remove(os.path.join(output_dir, f"frame_{t}.png"))
    
    print(f"✅ Saved comparison GIF: {gif_path}")


In [ ]:


create_condition_comparison_gif(archetype_node_activity_best, centers, node_codes,
                                arch_idx=17, n_timepoints=300, colors=colors)


In [ ]:

create_condition_comparison_gif(archetype_node_activity_compare, centers, node_codes,
                                arch_idx=17, n_timepoints=300, colors=colors)